# GraphSentinel v2 — Training Notebook

**Graph-based network intrusion detection.** Nodes are hosts, edges are flows.

This notebook is a thin driver. All logic lives in the `graphsentinel/` package
so it can be tested, versioned and imported by a service — a notebook cell is
not a deployment artefact.

---

## What changed from v1, and why

| # | v1 | v2 |
|---|---|---|
| 1 | Nodes = flow rows; edges invented from row adjacency and shared ports | Nodes = **IP addresses**, edges = **flows** carrying features |
| 2 | Fixed 500-flow windows | **Time** windows + persistent per-host GRU memory across windows |
| 3 | `SAGEConv(aggr='mean')` + scalar class weights on CE | **GATv2 attention** (or median) + **focal loss** + degree reweighting |
| 4 | Binary output (`out_channels=2`) | **6-way multi-class**, node head **and** per-flow edge head |
| 5 | Offline CSVs only | **Streaming engine** + replay/CICFlowMeter/scapy sources + FastAPI service |
| 6 | `port / 65535.0` | **Learned port embeddings** over a curated vocabulary |
| 7 | `in_channels=7` frozen by a backend import | **Model card + HTTP service**; compat shim for migration |
| 8 | `scaler.pkl` fitted once offline | **EMA scaler** with drift alarms and anti-poisoning clamps |
| 9 | Per-file 70/15/15 → leakage | **Four split protocols**, none of which cuts an attack burst in half |
| 10 | 3 bare SAGEConv layers → over-smoothing | **2 layers + residuals + Jumping Knowledge** |
| 11 | Per-row Python loop (~8 k flows/s) | **Vectorised** (~500 k flows/s, measured — 80x) |
| 12 | Raw logit, unusable by SDN | **Per-flow verdicts → OpenFlow rules** with 5-tuple matches |

---

## ⚠️ Before you run anything: check which CICIDS2017 you have

CICIDS2017 ships in two distributions **with identical filenames**:

- `MachineLearningCVE/` — 79 columns, starts with `' Destination Port'`.
  **No IP addresses.** This is what v1 was pointed at, and it is why v1 ended up
  building a sequence graph — there were no addresses in the data to make nodes
  out of.
- `TrafficLabelling_/` (inside `GeneratedLabelledFlows.zip`) — 85 columns,
  starts with `Flow ID, Source IP, Source Port, Destination IP, ...`

**You need the second one.** Download from
<https://www.unb.ca/cic/datasets/ids-2017.html>, take `GeneratedLabelledFlows.zip`
(not `MachineLearningCSV.zip`), and drop the same five filenames into the same
Drive folder. Section 2 will tell you immediately if you have the wrong one.

---
## 1 — Environment

In [ ]:
# Colab: PyG installs cleanly from PyPI on torch >= 2.1. The torch-scatter /
# torch-sparse wheels the old notebook chased are no longer required.
!pip install -q torch-geometric pyarrow
!pip install -q onnx onnxscript onnxruntime   # optional, for ONNX export

import torch, torch_geometric
print("torch          :", torch.__version__)
print("torch-geometric:", torch_geometric.__version__)
print("cuda           :", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/GraphSentinel"

# Put the graphsentinel/ package next to your data (upload it, or git clone).
import sys, os
sys.path.insert(0, BASE)          # so `import graphsentinel` works
os.makedirs(BASE, exist_ok=True)
print(sorted(os.listdir(BASE)))

---
## 2 — Configuration and dataset check

This cell fails loudly and with instructions if you have the wrong CICIDS2017 distribution.

In [ ]:
from graphsentinel.config import Config, CLASS_NAMES
from graphsentinel.data.schema import validate_dataset, SchemaError

cfg = Config()
cfg.base_dir = BASE

# --- the knobs worth touching --------------------------------------------
cfg.graph.window_seconds       = 60      # time window, not flow count
cfg.data.split_strategy        = "episode"   # see Section 3
cfg.model.conv_type            = "gatv2"     # gatv2 | sage_median | sage_max
cfg.model.num_layers           = 2
cfg.model.use_memory           = True
cfg.model.memory_capacity      = 262_144     # see docs/MEMORY.md for the maths
cfg.train.epochs               = 60
cfg.train.learning_rate        = 2e-3

cfg.ensure_dirs()
cfg.save(cfg.log_path / "config.json")
print("classes:", CLASS_NAMES)

try:
    validate_dataset(cfg.dataset_path, cfg.data.csv_files, require_ips=True)
except SchemaError as e:
    print(e)
    raise

---
## 3 — Choosing a split protocol

This is the decision that determines whether your reported numbers mean
anything. v1 took the first 70 % of each CSV for train and the last 15 % for
test — and since each CICIDS2017 file contains **one long attack**, the
beginning and the end of the *same* burst ended up on both sides. A model can
score 0.99 there by recognising that specific burst.

Four protocols, measuring four different things. **Report more than one.**

| protocol | what it measures | caveat on CICIDS2017 |
|---|---|---|
| `episode` *(default)* | whole attack bursts assigned to one split | most CICIDS2017 attacks are a single burst, so it falls back to a chronological cut with a dead zone — and says so |
| `temporal` | generalisation to *future* traffic | degenerates into a partial zero-day test: DDoS and PortScan only occur in the last hours, so they never appear in train |
| `host_holdout` | generalisation to a **new attacker** running the same technique | PortScan has one scanner IP, so that class cannot be host-split |
| `attack_holdout` | true **zero-day**: a family never seen in training | the held-out class cannot be *classified*, only flagged anomalous |

`episode` is the default because it is the only one under which a 6-way
classifier can actually be trained on this dataset. Be honest about its
limitation: where it falls back, the number is *within-episode* generalisation.
The zero-day claim has to come from `attack_holdout` — run it in Section 8.

---
## 4 — Build splits and graphs

In [ ]:
from graphsentinel.train import prepare_graphs
from graphsentinel.data.graph_builder import summarise_graphs
import json

graphs = prepare_graphs(cfg, force=False)   # force=True to rebuild the cache

for name, gs in graphs.items():
    print(f"\n{name}: {len(gs)} graphs")
    print(json.dumps(summarise_graphs(gs, cfg.model.num_classes), indent=2))

### Sanity check: does the graph actually encode attack topology?

Worth two minutes. If a port scanner does not stand out on port entropy, the
graph is wrong and no amount of model tuning will fix it.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from graphsentinel.data.graph_builder import NODE_FEATURE_NAMES
from graphsentinel.config import CLASS_NAMES

feats = {n: [] for n in ("dst_port_entropy", "log_unique_src_ips",
                         "log_unique_dst_ips", "mean_inter_flow_dt")}
labels = []
for g in graphs["train"]:
    labels.append(g.y.numpy())
    for n in feats:
        feats[n].append(g.x[:, NODE_FEATURE_NAMES.index(n)].numpy())
labels = np.concatenate(labels)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (name, vals) in zip(axes, feats.items()):
    vals = np.concatenate(vals)
    data = [vals[labels == i] for i in range(len(CLASS_NAMES))]
    data = [d if len(d) else np.array([0.0]) for d in data]
    ax.boxplot(data, tick_labels=[c[:6] for c in CLASS_NAMES], showfliers=False)
    ax.set_title(name); ax.tick_params(axis="x", rotation=45); ax.grid(alpha=.3)
plt.suptitle("Structural node features by class — separation here is the whole point",
             fontweight="bold")
plt.tight_layout(); plt.show()

---
## 5 — Train

No `input()` prompt anywhere: v1 blocked on `input("Resume? [y/n]")` inside the training cell, which hangs any unattended run forever. Resume is a flag.

In [ ]:
from graphsentinel.train import train

report = train(cfg, resume=True, force_rebuild=False)

---
## 6 — Curves

Macro F1 is the headline, not weighted F1. Weighted F1 weights DoS Hulk (167 k rows) at 111x Botnet (1.5 k) — a model that never detects a single botnet can still post 0.95.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from graphsentinel.config import CLASS_NAMES

hist = pd.read_csv(cfg.log_path / "training_log.csv")
fig, axes = plt.subplots(1, 3, figsize=(19, 5))

axes[0].plot(hist.epoch, hist.train_loss, lw=2)
axes[0].set_title("Train loss (focal + edge + consistency)")
axes[0].set_xlabel("epoch"); axes[0].grid(alpha=.3)

for k, lbl in (("node_macro_f1", "node macro F1"),
               ("edge_macro_f1", "edge macro F1"),
               ("node_binary_pr_auc", "binary PR-AUC")):
    if k in hist: axes[1].plot(hist.epoch, hist[k], lw=2, label=lbl)
axes[1].set_title("Validation"); axes[1].legend(); axes[1].grid(alpha=.3)
axes[1].set_ylim(0, 1.02); axes[1].set_xlabel("epoch")

for c in CLASS_NAMES:
    k = f"node_f1_{c}"
    if k in hist: axes[2].plot(hist.epoch, hist[k], lw=1.8, label=c)
axes[2].set_title("Per-class F1 — watch the minority classes")
axes[2].legend(fontsize=8); axes[2].grid(alpha=.3); axes[2].set_ylim(0, 1.02)
axes[2].set_xlabel("epoch")

plt.tight_layout(); plt.savefig(cfg.log_path / "training_curves.png", dpi=150)
plt.show()

---
## 7 — Test results, confusion, and the evasion check

In [ ]:
import json, numpy as np, seaborn as sns, matplotlib.pyplot as plt
from graphsentinel.config import CLASS_NAMES

rep = json.load(open(cfg.log_path / "test_report.json"))
m = rep["metrics"]

print("=" * 62)
print(f"  TEST — split protocol: {cfg.data.split_strategy}")
print("=" * 62)
for key in ("node_macro_f1", "edge_macro_f1", "node_binary_pr_auc",
            "node_recall_at_fpr_0.01", "node_recall_at_fpr_0.001"):
    if key in m: print(f"  {key:<30s} {m[key]:.4f}")
print("\n  per-class node F1:")
for c in CLASS_NAMES:
    print(f"    {c:<10s} {m.get(f'node_f1_{c}', float('nan')):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, kind in zip(axes, ("node", "edge")):
    cm = np.array(rep[f"{kind}_confusion"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_title(f"{kind}-level confusion"); ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.tight_layout(); plt.show()

### Evasion ablation

`Flow Bytes/s` and `Flow Duration` are under the attacker's direct control —
rate-limit the payload, fragment the packets, and those columns say whatever the
attacker wants. This zeroes them and re-measures.

A model that **collapses** here was reading spoofable headers. A model that
mostly **holds** is reading topology, which an attacker cannot fake without
abandoning the attack: a port scan that stops contacting many ports has stopped
being a port scan.

In [ ]:
abl = rep["evasion_ablation"]
print("metric                          baseline   ablated     delta")
for k, v in abl["delta"].items():
    b, a = abl["baseline"].get(k, float("nan")), abl["ablated"].get(k, float("nan"))
    print(f"  {k:<30s} {b:7.4f}   {a:7.4f}   {v:+7.4f}")
print("\nSmall deltas = the model is reading structure, not volume.")

---
## 8 — Zero-day evaluation (the number the review actually asked for)

Retrain with one attack family removed entirely, then measure whether the held-out family is still flagged as *something*. A binary threat score can catch an unknown attack; a class label cannot name it.

In [ ]:
from graphsentinel.config import Config
from graphsentinel.train import train

zero_day = {}
for family in ["Botnet", "PortScan"]:
    c = Config.from_dict(cfg.to_dict())
    c.data.split_strategy   = "attack_holdout"
    c.data.holdout_attacks  = [family]
    c.base_dir              = f"{BASE}/zeroday_{family}"
    c.train.epochs          = 30
    c.ensure_dirs()
    print(f"\n{'='*62}\n  HOLDING OUT: {family}\n{'='*62}")
    r = train(c, resume=False, force_rebuild=True, verbose=True)
    zero_day[family] = r["metrics"]

print("\n  zero-day detection (binary — can it flag an attack it never saw?)")
for fam, mm in zero_day.items():
    print(f"    {fam:<10s} PR-AUC {mm.get('node_binary_pr_auc', float('nan')):.4f}"
          f"   recall@1%FPR {mm.get('node_recall_at_fpr_0.01', float('nan')):.4f}")

---
## 9 — Export for the backend

Writes `weights.pt`, `model_card.json`, EMA scaler seeds, TorchScript and ONNX. The backend reads the card and calls the service; it never imports a model class.

In [ ]:
import torch, numpy as np
from graphsentinel.export import export_all, verify_export
from graphsentinel.models.net import build_model
from graphsentinel.inference.ema_scaler import EMAScaler
from graphsentinel.data.graph_builder import EDGE_FEATURE_NAMES, NODE_FEATURE_NAMES

model = build_model(cfg)
best = torch.load(cfg.checkpoint_path / "best.pt", map_location="cpu", weights_only=False)
model.load_state_dict(best["model"])

# Warm-start the streaming scaler from the TRAINING feature distribution.
edge_x = np.concatenate([g.edge_attr.numpy() for g in graphs["train"][:200]])
node_x = np.concatenate([g.x.numpy()          for g in graphs["train"][:200]])
edge_scaler = EMAScaler.from_training(edge_x, EDGE_FEATURE_NAMES)
node_scaler = EMAScaler.from_training(node_x, NODE_FEATURE_NAMES)

status = export_all(cfg, model, cfg.model_path, edge_scaler, node_scaler,
                    metrics=rep["metrics"])
print("\nverification:", verify_export(cfg.model_path, model))

---
## 10 — Streaming inference and SDN rules

Replay a CSV through the same code path production uses. `speed=1.0` replays in real time, which is how you find out whether the pipeline keeps up *before* pointing it at a span port.

In [ ]:
from graphsentinel.inference.engine import InferenceEngine
from graphsentinel.inference.capture import CSVReplaySource, run_pipeline

engine = InferenceEngine.from_artifacts(cfg.model_path, threat_threshold=0.75)
source = CSVReplaySource(
    cfg.dataset_path / "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    speed=0.0, limit=60_000,          # speed=1.0 for real-time
)

results = []
run_pipeline(source, engine, batch_size=2048, on_result=results.append)

print(f"windows: {len(results)}   flows: {sum(r.n_flows for r in results):,}")
print(f"median window latency: "
      f"{np.median([r.latency_ms for r in results]):.1f} ms "
      f"(budget: {cfg.graph.window_seconds * 1000} ms)")

for r in results[:20]:
    for d in r.detections[:3]:
        print(f"  {d.ip:<16s} {d.attack_class:<10s} {d.threat_score:.3f} "
              f"out={d.out_degree:<6d} in={d.in_degree}")

In [ ]:
# The rules an SDN controller can actually install
n = 0
for r in results:
    for rule in r.rules:
        print(f"{rule.attack_class:<10s} {rule.src_ip:>15s} -> {rule.dst_ip:<15s} "
              f":{rule.dst_port:<6d} proto={rule.protocol}  {rule.action:<20s} "
              f"conf={rule.confidence:.3f}")
        print("   ", rule.to_ovs_ofctl())
        n += 1
        if n >= 12: break
    if n >= 12: break
print(f"\ntotal rules generated: {sum(len(r.rules) for r in results)}")
print("NOTE: translator defaults to dry_run=True. Set allow_networks for your "
      "gateways/DNS/controller BEFORE enabling enforcement.")

---
## 11 — Hand-off

Give Sairaj the **`models/` directory** and **`docs/BACKEND_CONTRACT.md`**.

Do **not** hand over `graphsage_weights.pt` + `scaler.pkl` and a feature list in
a comment — that arrangement is what pinned the architecture in the first place.

Migration is three steps and only the last one requires his attention:

1. `from graphsentinel.models.compat import GraphSAGEClassifier` — still works,
   warns, nothing breaks.
2. `LegacyBinaryAdapter(v2_model).predict_proba(graph)` — new model, v1 method
   names, his `THREAT_THRESHOLD` logic untouched.
3. `POST /flows` to the inference service — no model import at all, and after
   this the ML side can change architecture freely.

```bash
GRAPHSENTINEL_MODEL_DIR=models/ \
  uvicorn graphsentinel.inference.service:app --host 0.0.0.0 --port 8080
```